In [ ]:
import numpy as np
import pandas as pd
import yaml
import os
from outbreak_data import outbreak_data as od
from outbreak_data import authenticate_user
from outbreak_tools import crumbs


authenticate_user.authenticate_new_user()

: 

In [ ]:
# Load data
lineage_yml = yaml.safe_load('lineages.yml')
print(lineage_yml)
LINEAGES = []

for file in os.listdir('samples/'):
    LINEAGES.append(file.split('_')[1].split('.X')[0])

lineage_key = crumbs.get_alias_key()

AY_muts = od.lineage_mutations(pango_lin='B.1.617.2', descendants=True, lineage_key=lineage_key).index.to_list()
BA_muts = od.lineage_mutations(pango_lin='B.1.1.529', descendants=True, lineage_key=lineage_key).index.to_list()

# Get the intersection of AY and BA mutations
shared_muts = list(set(AY_muts).intersection(set(BA_muts)))


print(all_covariants)


In [ ]:
import json
all_covariants = pd.read_csv('all_covariants.csv')

lineage_key = crumbs.get_alias_key()

# Calculate posterior probability of observing Delta given the observed mutations
def _multiquery_to_df(data):
    return pd.concat([pd.DataFrame(v).assign(query=k) for k,v in data['results'].items()], axis=0)

START_DATE = '2023-01-01'
END_DATE = '2023-12-31'

delta_prevalence = od.lineage_cl_prevalence('B.1.617.2', descendants=True, location='USA', datemin=START_DATE, datemax=END_DATE, lineage_key=lineage_key)
all_sequences = delta_prevalence['total_count'].sum()
P_delta = delta_prevalence['lineage_count'].sum() / all_sequences

print('All Sequences:', all_sequences)
print('Delta Sequences:', delta_prevalence['lineage_count'].sum())

all_covariants['P(M|D)'] = 0.0
all_covariants['P(D)'] = P_delta
all_covariants['P(M)'] = 0.0
all_covariants['P(D|M)'] = 0.0

for index, row in all_covariants.iterrows():

    cluster = row['Covariants']
    cluster = cluster.replace('[','').replace(']','').replace('\'','').split(', ')
    print(cluster)
    try:

        mut_data = od.lineage_cl_prevalence('.', descendants=True, mutations=cluster, location='USA', datemin=START_DATE, datemax=END_DATE, lineage_key=lineage_key)
        formatted_cluster = ' AND '.join([f"mutations:{m.replace(':','?')}" for m in cluster])
        print(type(cluster))
        print(formatted_cluster)
        response = od.requests.get(f'https://api.outbreak.info/genomics/prevalence-by-location?lineages=None&q=pangolin_lineage_crumbs:*;B.1.617;* AND {formatted_cluster}&cumulative=false&min_date={START_DATE}&max_date={END_DATE}&location_id=USA',
                                   headers=od._get_user_authentication())
        delta_mut_data = _multiquery_to_df(response.json())

        mut_data['total_count']
        delta_mut_data['total_count']
    except Exception as e:
        continue


    P_mutation_clusters_given_delta = delta_mut_data['lineage_count'].sum() / delta_prevalence['lineage_count'].sum()
    P_mutation = mut_data['lineage_count'].sum() / mut_data['total_count'].sum()
    P_delta_given_mutation_clusters = P_mutation_clusters_given_delta * P_delta / P_mutation
          
    all_covariants.loc[index,'P(D|M)'] = P_delta_given_mutation_clusters
    all_covariants.loc[index,'P(M)'] = P_mutation
    all_covariants.loc[index,'P(M|D)'] = P_mutation_clusters_given_delta


before = len(all_covariants)
after = len(all_covariants)
print(f'Dropped {before - after} clusters with no clinical data')
all_covariants.to_csv('all_covariants_probabilities_2023.csv', index=False)


In [ ]:
df = pd.read_csv('all_covariants_probabilities_2023.csv')
df = df[df['P(D|M)'] > 0.95]
df.to_csv('all_covariants_2023_filtered.csv', index=False)